In [76]:
import pandas as pd




In [77]:
consolidated_pairs = pd.read_csv('consolidated_pairs.csv')
print(f"Total rows loaded: {len(consolidated_pairs)}")

# Filter out rows with NaN pair_id (invalid rows)
consolidated_pairs = consolidated_pairs[consolidated_pairs['pair_id'].notna()].copy()
print(f"Valid pairs (after filtering): {len(consolidated_pairs)}")
print("\nFirst few rows:")
print(consolidated_pairs.head())

Total rows loaded: 34
Valid pairs (after filtering): 32

First few rows:
                      pair_id session_code director_participant_code  \
0  3kk7u0fy_08dnx2dx_ljtwijvs     3kk7u0fy                  ljtwijvs   
1  3kk7u0fy_2ifxj59u_qhyb02an     3kk7u0fy                  2ifxj59u   
2  3kk7u0fy_2ivt4mpj_g7urxzlh     3kk7u0fy                  2ivt4mpj   
3  3kk7u0fy_ddls1zzn_ldgryt73     3kk7u0fy                  ldgryt73   
4  3kk7u0fy_ecjbn129_fkm1onm5     3kk7u0fy                  fkm1onm5   

  director_time_started_utc      director_prolific_id  \
0                   24:36.3  6562177a48308eebbb6359ea   
1                   38:01.1  67e8e0c0be5802df7311d109   
2                   30:14.5  5fd66ce8aec66457ff73d743   
3                   30:12.2  666dbb1357a8cfb6025951df   
4                   03:12.4  67ace3fc5892a4d45463156b   

        matcher_prolific_id director_experiment_start_time  \
0  5c28e9dfda51990001e3204b     2025-11-24T18:24:43.139397   
1  67dac9fec33b7784afd3d08b

In [78]:
import re
from datetime import datetime

def get_first_and_last_timestamps(chat_transcript):
    if pd.isnull(chat_transcript) or not isinstance(chat_transcript, str):
        return None, None
    # Expected timestamp format: [HH:MM:SS]
    timestamps = re.findall(r'\[(\d{2}:\d{2}:\d{2})\]', chat_transcript)
    if not timestamps:
        return None, None
    return timestamps[0], timestamps[-1]

def get_duration_seconds(start_ts, end_ts):
    if not start_ts or not end_ts:
        return None
    fmt = "%H:%M:%S"
    try:
        t1 = datetime.strptime(start_ts, fmt)
        t2 = datetime.strptime(end_ts, fmt)
        delta = (t2 - t1).total_seconds()
        # handle possible day wraparound
        if delta < 0:
            delta += 24 * 60 * 60
        return delta
    except Exception:
        return None

def seconds_to_hhmmss(seconds):
    """Convert seconds to HH:MM:SS format"""
    if seconds is None or pd.isnull(seconds):
        return None
    try:
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    except Exception:
        return None

# Process each round and add time columns to consolidated_pairs (keeping wide format)
for round_num in range(1, 5):
    round_times = []
    
    for idx, row in consolidated_pairs.iterrows():
        # Get chat transcript for this round
        matcher_col = f'round{round_num}_matcher_chat_transcript'
        director_col = f'round{round_num}_director_chat_transcript'
        
        transcript = row.get(matcher_col, "")
        if pd.isnull(transcript) or (isinstance(transcript, str) and transcript.strip() == ''):
            transcript = row.get(director_col, "")
        
        # Extract timestamps and calculate duration
        first_ts, last_ts = get_first_and_last_timestamps(transcript)
        duration_seconds = get_duration_seconds(first_ts, last_ts)
        duration_hhmmss = seconds_to_hhmmss(duration_seconds)
        round_times.append(duration_hhmmss)
    
    # Add the column for this round
    consolidated_pairs[f'round{round_num}_time'] = round_times

# Show the new time columns
time_cols = ['pair_id'] + [f'round{i}_time' for i in range(1, 5)]
print(consolidated_pairs[time_cols].head())


                      pair_id round1_time round2_time round3_time round4_time
0  3kk7u0fy_08dnx2dx_ljtwijvs    00:18:43    00:17:47    00:10:33    00:13:27
1  3kk7u0fy_2ifxj59u_qhyb02an    00:15:30    00:10:06    00:03:21    00:03:42
2  3kk7u0fy_2ivt4mpj_g7urxzlh    00:12:43    00:13:44    00:06:54    00:04:33
3  3kk7u0fy_ddls1zzn_ldgryt73    00:17:43    00:13:02    00:15:19    00:05:31
4  3kk7u0fy_ecjbn129_fkm1onm5    00:07:55    00:06:04    00:09:12    00:05:06


In [79]:
consolidated_pairs.to_csv('consolidated_pairs_with_times.csv', index=False)

In [80]:
import re
from collections import defaultdict
from datetime import datetime

def parse_transcript(transcript_text):
    """
    Parse a chat transcript and extract language units with timestamps.
    Returns: {
        'turns': list of turns (each turn is a list of (timestamp, message) tuples),
        'utterances': list of all (timestamp, message) tuples,
        'words': list of word counts per utterance
    }
    A turn is all contributions from one DP without interruption by the other DP.
    An utterance is a single message (up to carriage return).
    """
    if pd.isnull(transcript_text) or not isinstance(transcript_text, str) or not transcript_text.strip():
        return {'turns': [], 'utterances': [], 'words': []}
    
    # Pattern to match: [HH:MM:SS] speaker: message
    # Messages can span multiple lines until the next timestamp
    pattern = r'\[(\d{2}:\d{2}:\d{2})\]\s*(director|matcher):\s*(.*?)(?=\[\d{2}:\d{2}:\d{2}\]|$)'
    
    matches = re.findall(pattern, transcript_text, re.DOTALL)
    
    turns = []
    utterances = []
    words = []
    
    current_turn = []
    current_speaker = None
    
    for timestamp, speaker, message in matches:
        message = message.strip()
        if not message:
            continue
        
        # Check if this is a new turn (different speaker)
        if current_speaker is not None and speaker != current_speaker:
            # End the current turn
            if current_turn:
                turns.append(current_turn)
            current_turn = []
        
        # Add this utterance with timestamp
        current_turn.append((timestamp, message))
        utterances.append((timestamp, message))
        
        # Count words in this utterance
        words_in_utterance = len(re.findall(r'\b\w+\b', message))
        words.extend([words_in_utterance])  # Store word count per utterance for easier calculation
        
        current_speaker = speaker
    
    # Don't forget the last turn
    if current_turn:
        turns.append(current_turn)
    
    return {
        'turns': turns,
        'utterances': utterances,
        'words': words  # This is word counts per utterance
    }

def timestamp_to_seconds(ts_str):
    """Convert HH:MM:SS timestamp to seconds since start of day."""
    try:
        parts = ts_str.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    except:
        return None

def calculate_time_difference(start_ts, end_ts):
    """Calculate time difference in seconds between two timestamps."""
    start_sec = timestamp_to_seconds(start_ts)
    end_sec = timestamp_to_seconds(end_ts)
    if start_sec is None or end_sec is None:
        return None
    # Handle day wraparound
    if end_sec < start_sec:
        end_sec += 24 * 3600
    return end_sec - start_sec

def seconds_to_hhmmss(seconds):
    """Convert seconds to HH:MM:SS format string."""
    if seconds is None or pd.isnull(seconds):
        return None
    try:
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    except (TypeError, ValueError):
        return None

def calculate_metrics_for_round(transcript_text):
    """Calculate all language unit metrics for a single round, including time metrics."""
    parsed = parse_transcript(transcript_text)
    
    turns = parsed['turns']
    utterances = parsed['utterances']
    word_counts = parsed['words']  # List of word counts per utterance
    
    total_words = sum(word_counts)
    num_turns = len(turns)
    num_utterances = len(utterances)
    
    # Calculate round duration (from first to last timestamp)
    round_duration_seconds = None
    if utterances:
        first_ts = utterances[0][0]
        last_ts = utterances[-1][0]
        round_duration_seconds = calculate_time_difference(first_ts, last_ts)
    
    # Calculate turn durations
    turn_durations = []
    for turn in turns:
        if len(turn) > 0:
            turn_start_ts = turn[0][0]
            turn_end_ts = turn[-1][0]
            turn_duration = calculate_time_difference(turn_start_ts, turn_end_ts)
            if turn_duration is not None:
                turn_durations.append(turn_duration)
    
    # Calculate utterance durations (inter-utterance intervals)
    # This measures the time between consecutive utterances across the entire round
    # Note: This includes intervals both WITHIN turns and BETWEEN turns (when speakers switch)
    utterance_durations = []
    for i, (ts, msg) in enumerate(utterances):
        if i < len(utterances) - 1:
            # Duration from this utterance to the next
            next_ts = utterances[i + 1][0]
            duration = calculate_time_difference(ts, next_ts)
            if duration is not None:
                utterance_durations.append(duration)
    # Note: We don't include the last utterance as it has no "next" utterance
    
    # Calculate average time metrics
    avg_turn_duration = sum(turn_durations) / len(turn_durations) if turn_durations else None
    avg_utterance_duration = sum(utterance_durations) / len(utterance_durations) if utterance_durations else None
    
    # IMPORTANT NOTE: avg_turn_duration vs avg_utterance_duration
    # - avg_turn_duration: Time span WITHIN a turn (from first to last utterance in that turn)
    #   If a turn has only 1 utterance, duration = 0 seconds
    # - avg_utterance_duration: Time BETWEEN consecutive utterances (inter-utterance interval)
    #   This includes gaps both within turns AND between turns (when speakers switch)
    #   So avg_utterance_duration can be longer than avg_turn_duration if:
    #   - Most turns have only 1 utterance (making turn durations short)
    #   - There are long gaps between turns (when the other speaker responds)
    
    # Calculate metrics
    metrics = {
        'words': total_words,
        'turns': num_turns,
        'utterances': num_utterances,
        'words_per_round': total_words,
        'words_per_turn': total_words / num_turns if num_turns > 0 else 0,
        'words_per_utterance': total_words / num_utterances if num_utterances > 0 else 0,
        'utterances_per_turn': num_utterances / num_turns if num_turns > 0 else 0,
        'utterances_per_round': num_utterances,
        'turns_per_round': num_turns,
        'round_duration_seconds': round_duration_seconds,
        'avg_turn_duration_seconds': avg_turn_duration,
        'avg_utterance_duration_seconds': avg_utterance_duration,
        # Formatted time strings
        'round_duration': seconds_to_hhmmss(round_duration_seconds),
        'avg_turn_duration': seconds_to_hhmmss(avg_turn_duration),
        'avg_utterance_duration': seconds_to_hhmmss(avg_utterance_duration)
    }
    
    return metrics

# Process all rounds for all pairs (only valid pairs with non-null pair_id)
round_metrics = []

for idx, row in consolidated_pairs.iterrows():
    pair_id = row['pair_id']
    
    # Skip if pair_id is NaN (shouldn't happen after filtering, but double-check)
    if pd.isna(pair_id):
        continue
    
    for round_num in range(1, 5):
        # Get transcript (use director or matcher - they're duplicates)
        director_col = f'round{round_num}_director_chat_transcript'
        matcher_col = f'round{round_num}_matcher_chat_transcript'
        
        transcript = row.get(director_col, '')
        if pd.isnull(transcript) or (isinstance(transcript, str) and transcript.strip() == ''):
            transcript = row.get(matcher_col, '')
        
        # Get accuracy for this round
        accuracy_col = f'round{round_num}_matcher_sequence_accuracy'
        accuracy = row.get(accuracy_col, None)
        
        # Calculate metrics
        metrics = calculate_metrics_for_round(transcript)
        metrics['pair_id'] = pair_id
        metrics['round'] = round_num
        metrics['accuracy'] = accuracy
        
        round_metrics.append(metrics)

# Convert to DataFrame
metrics_df = pd.DataFrame(round_metrics)
print("Sample metrics:")
print(metrics_df.head(10))


Sample metrics:
   words  turns  utterances  words_per_round  words_per_turn  \
0    374     32          33              374       11.687500   
1    352     31          34              352       11.354839   
2    268     26          26              268       10.307692   
3    299     27          27              299       11.074074   
4    299     19          38              299       15.736842   
5    216     14          30              216       15.428571   
6    117      6          19              117       19.500000   
7    127      5          18              127       25.400000   
8    341     47          53              341        7.255319   
9    350     47          56              350        7.446809   

   words_per_utterance  utterances_per_turn  utterances_per_round  \
0            11.333333             1.031250                    33   
1            10.352941             1.096774                    34   
2            10.307692             1.000000                    26   
3  

In [81]:
# Calculate average accuracy per round across all dialogs
avg_accuracy_by_round = metrics_df.groupby('round')['accuracy'].mean()
std_accuracy_by_round = metrics_df.groupby('round')['accuracy'].std()

print("Average Accuracy per Round (Mean ± SD):")
for round_num in range(1, 5):
    mean_val = avg_accuracy_by_round[round_num]
    std_val = std_accuracy_by_round[round_num]
    print(f"Round {round_num}: {mean_val:.2f}% ± {std_val:.2f}%")

print("\nExpectation for humans: gets higher from round to round")
print("Expectation for AI: does not change")


Average Accuracy per Round (Mean ± SD):
Round 1: 80.21% ± 19.49%
Round 2: 86.46% ± 19.94%
Round 3: 90.36% ± 19.53%
Round 4: 92.19% ± 15.83%

Expectation for humans: gets higher from round to round
Expectation for AI: does not change


In [82]:
# Calculate average metrics per round
metric_columns = [
    'words_per_round',
    'words_per_turn', 
    'words_per_utterance',
    'utterances_per_turn',
    'utterances_per_round',
    'turns_per_round'
]

avg_metrics_by_round = metrics_df.groupby('round')[metric_columns].mean()
std_metrics_by_round = metrics_df.groupby('round')[metric_columns].std()

print("Average Language Metrics per Round (Mean ± SD):")
print("\n" + "="*80)
for metric in metric_columns:
    print(f"\n{metric.replace('_', ' ').title()}:")
    for round_num in range(1, 5):
        mean_val = avg_metrics_by_round.loc[round_num, metric]
        std_val = std_metrics_by_round.loc[round_num, metric]
        print(f"  Round {round_num}: {mean_val:.2f} ± {std_val:.2f}")


Average Language Metrics per Round (Mean ± SD):


Words Per Round:
  Round 1: 449.47 ± 345.76
  Round 2: 304.91 ± 174.97
  Round 3: 252.41 ± 156.86
  Round 4: 221.94 ± 206.58

Words Per Turn:
  Round 1: 15.78 ± 13.35
  Round 2: 19.44 ± 29.14
  Round 3: 17.81 ± 24.11
  Round 4: 19.60 ± 29.28

Words Per Utterance:
  Round 1: 9.77 ± 5.02
  Round 2: 8.83 ± 4.58
  Round 3: 7.87 ± 3.78
  Round 4: 6.95 ± 3.80

Utterances Per Turn:
  Round 1: 1.62 ± 1.18
  Round 2: 1.88 ± 1.62
  Round 3: 2.00 ± 1.60
  Round 4: 2.60 ± 3.24

Utterances Per Round:
  Round 1: 47.88 ± 37.23
  Round 2: 37.78 ± 21.81
  Round 3: 33.47 ± 19.66
  Round 4: 33.50 ± 30.68

Turns Per Round:
  Round 1: 35.53 ± 25.95
  Round 2: 28.22 ± 16.26
  Round 3: 24.19 ± 15.33
  Round 4: 23.56 ± 17.20


In [ ]:
# Summary statistics
print("Summary Statistics by Round (Mean ± SD):")
print("\n" + "="*60)
for round_num in range(1, 5):
    round_data = metrics_df[metrics_df['round'] == round_num]
    print(f"\nRound {round_num}:")
    print(f"  Number of dialogs: {len(round_data)}")
    print(f"ho  Average accuracy: {round_data['accuracy'].mean():.2f}% ± {round_data['accuracy'].std():.2f}%")
    print(f"  Average words per round: {round_data['words_per_round'].mean():.1f} ± {round_data['words_per_round'].std():.1f}")
    print(f"  Average words per turn: {round_data['words_per_turn'].mean():.1f} ± {round_data['words_per_turn'].std():.1f}")
    print(f"  Average words per utterance: {round_data['words_per_utterance'].mean():.1f} ± {round_data['words_per_utterance'].std():.1f}")
    print(f"  Average utterances per turn: {round_data['utterances_per_turn'].mean():.2f} ± {round_data['utterances_per_turn'].std():.2f}")
    print(f"  Average utterances per round: {round_data['utterances_per_round'].mean():.1f} ± {round_data['utterances_per_round'].std():.1f}")
    print(f"  Average turns per round: {round_data['turns_per_round'].mean():.1f} ± {round_data['turns_per_round'].std():.1f}")
    
    # Time metrics (display in HH:MM:SS format)
    round_durations = round_data['round_duration_seconds'].dropna()
    turn_durations = round_data['avg_turn_duration_seconds'].dropna()
    utterance_durations = round_data['avg_utterance_duration_seconds'].dropna()
    
    if len(round_durations) > 0:
        avg_round_time = round_durations.mean()
        std_round_time = round_durations.std()
        print(f"  Average round duration: {seconds_to_hhmmss(avg_round_time)} ± {seconds_to_hhmmss(std_round_time)}")
    if len(turn_durations) > 0:
        avg_turn_time = turn_durations.mean()
        std_turn_time = turn_durations.std()
        print(f"  Average turn duration (within-turn span): {seconds_to_hhmmss(avg_turn_time)} ± {seconds_to_hhmmss(std_turn_time)}")
    if len(utterance_durations) > 0:
        avg_utterance_time = utterance_durations.mean()
        std_utterance_time = utterance_durations.std()
        print(f"  Average inter-utterance interval (between consecutive utterances): {seconds_to_hhmmss(avg_utterance_time)} ± {seconds_to_hhmmss(std_utterance_time)}")


Summary Statistics by Round (Mean ± SD):


Round 1:
  Number of dialogs: 32
  Average accuracy: 80.21% ± 19.49%
  Average words per round: 449.5 ± 345.8
  Average words per turn: 15.8 ± 13.4
  Average words per utterance: 9.8 ± 5.0
  Average utterances per turn: 1.62 ± 1.18
  Average utterances per round: 47.9 ± 37.2
  Average turns per round: 35.5 ± 26.0
  Average round duration: 00:18:53 ± 00:11:23
  Average turn duration (within-turn span): 00:00:18 ± 00:00:41
  Average inter-utterance interval (between consecutive utterances): 00:00:29 ± 00:00:15

Round 2:
  Number of dialogs: 32
  Average accuracy: 86.46% ± 19.94%
  Average words per round: 304.9 ± 175.0
  Average words per turn: 19.4 ± 29.1
  Average words per utterance: 8.8 ± 4.6
  Average utterances per turn: 1.88 ± 1.62
  Average utterances per round: 37.8 ± 21.8
  Average turns per round: 28.2 ± 16.3
  Average round duration: 00:12:05 ± 00:06:54
  Average turn duration (within-turn span): 00:00:27 ± 00:01:02
  Average inter-u

In [84]:
# Calculate average time metrics per round
time_metrics_by_round_mean = metrics_df.groupby('round').agg({
    'round_duration_seconds': 'mean',
    'avg_turn_duration_seconds': 'mean',
    'avg_utterance_duration_seconds': 'mean'
}).round(2)

time_metrics_by_round_std = metrics_df.groupby('round').agg({
    'round_duration_seconds': 'std',
    'avg_turn_duration_seconds': 'std',
    'avg_utterance_duration_seconds': 'std'
}).round(2)

print("Average Time Metrics per Round (Mean ± SD, HH:MM:SS format):")
print("\nRound | Round Duration (Mean ± SD) | Turn Duration (Mean ± SD) | Utterance Duration (Mean ± SD)")
print("-" * 100)
for round_num in range(1, 5):
    round_dur_mean = time_metrics_by_round_mean.loc[round_num, 'round_duration_seconds']
    round_dur_std = time_metrics_by_round_std.loc[round_num, 'round_duration_seconds']
    turn_dur_mean = time_metrics_by_round_mean.loc[round_num, 'avg_turn_duration_seconds']
    turn_dur_std = time_metrics_by_round_std.loc[round_num, 'avg_turn_duration_seconds']
    utt_dur_mean = time_metrics_by_round_mean.loc[round_num, 'avg_utterance_duration_seconds']
    utt_dur_std = time_metrics_by_round_std.loc[round_num, 'avg_utterance_duration_seconds']
    
    round_str = f"{seconds_to_hhmmss(round_dur_mean)} ± {seconds_to_hhmmss(round_dur_std)}"
    turn_str = f"{seconds_to_hhmmss(turn_dur_mean)} ± {seconds_to_hhmmss(turn_dur_std)}"
    utt_str = f"{seconds_to_hhmmss(utt_dur_mean)} ± {seconds_to_hhmmss(utt_dur_std)}"
    
    print(f"  {round_num}   | {round_str:>28} | {turn_str:>26} | {utt_str:>32}")

print("\nNote: Round Duration = total round time")
print("      Avg Turn Duration = average time span WITHIN a turn (from first to last utterance)")
print("      Avg Utterance Duration = average time BETWEEN consecutive utterances")
print("      (includes intervals both within turns AND between turns when speakers switch)")
print("\n      Why can utterance duration be longer than turn duration?")
print("      - If most turns have only 1 utterance, turn durations are short (~0s)")
print("      - Utterance durations include gaps BETWEEN turns (response time), which can be longer")

# Save the detailed metrics to CSV
metrics_df.to_csv('dialogue_metrics_by_round.csv', index=False)
print("\nSaved detailed metrics to 'dialogue_metrics_by_round.csv'")

# Save summary by round (include both seconds and formatted times, with std)
summary_by_round = pd.DataFrame({
    'round': range(1, 5),
    'avg_accuracy': avg_accuracy_by_round.values,
    'std_accuracy': std_accuracy_by_round.values,
    'avg_words_per_round': avg_metrics_by_round['words_per_round'].values,
    'std_words_per_round': std_metrics_by_round['words_per_round'].values,
    'avg_words_per_turn': avg_metrics_by_round['words_per_turn'].values,
    'std_words_per_turn': std_metrics_by_round['words_per_turn'].values,
    'avg_words_per_utterance': avg_metrics_by_round['words_per_utterance'].values,
    'std_words_per_utterance': std_metrics_by_round['words_per_utterance'].values,
    'avg_utterances_per_turn': avg_metrics_by_round['utterances_per_turn'].values,
    'std_utterances_per_turn': std_metrics_by_round['utterances_per_turn'].values,
    'avg_utterances_per_round': avg_metrics_by_round['utterances_per_round'].values,
    'std_utterances_per_round': std_metrics_by_round['utterances_per_round'].values,
    'avg_turns_per_round': avg_metrics_by_round['turns_per_round'].values,
    'std_turns_per_round': std_metrics_by_round['turns_per_round'].values,
    'avg_round_duration_seconds': time_metrics_by_round_mean['round_duration_seconds'].values,
    'std_round_duration_seconds': time_metrics_by_round_std['round_duration_seconds'].values,
    'avg_round_duration': [seconds_to_hhmmss(x) for x in time_metrics_by_round_mean['round_duration_seconds'].values],
    'std_round_duration': [seconds_to_hhmmss(x) for x in time_metrics_by_round_std['round_duration_seconds'].values],
    'avg_turn_duration_seconds': time_metrics_by_round_mean['avg_turn_duration_seconds'].values,
    'std_turn_duration_seconds': time_metrics_by_round_std['avg_turn_duration_seconds'].values,
    'avg_turn_duration': [seconds_to_hhmmss(x) for x in time_metrics_by_round_mean['avg_turn_duration_seconds'].values],
    'std_turn_duration': [seconds_to_hhmmss(x) for x in time_metrics_by_round_std['avg_turn_duration_seconds'].values],
    'avg_utterance_duration_seconds': time_metrics_by_round_mean['avg_utterance_duration_seconds'].values,
    'std_utterance_duration_seconds': time_metrics_by_round_std['avg_utterance_duration_seconds'].values,
    'avg_utterance_duration': [seconds_to_hhmmss(x) for x in time_metrics_by_round_mean['avg_utterance_duration_seconds'].values],
    'std_utterance_duration': [seconds_to_hhmmss(x) for x in time_metrics_by_round_std['avg_utterance_duration_seconds'].values]
})
summary_by_round.to_csv('dialogue_metrics_summary_by_round.csv', index=False)
print("Saved summary metrics to 'dialogue_metrics_summary_by_round.csv'")


Average Time Metrics per Round (Mean ± SD, HH:MM:SS format):

Round | Round Duration (Mean ± SD) | Turn Duration (Mean ± SD) | Utterance Duration (Mean ± SD)
----------------------------------------------------------------------------------------------------
  1   |          00:18:53 ± 00:11:23 |        00:00:18 ± 00:00:41 |              00:00:29 ± 00:00:15
  2   |          00:12:05 ± 00:06:54 |        00:00:27 ± 00:01:02 |              00:00:22 ± 00:00:10
  3   |          00:09:10 ± 00:04:46 |        00:00:25 ± 00:00:50 |              00:00:19 ± 00:00:11
  4   |          00:07:20 ± 00:05:03 |        00:00:30 ± 00:01:04 |              00:00:15 ± 00:00:08

Note: Round Duration = total round time
      Avg Turn Duration = average time span WITHIN a turn (from first to last utterance)
      Avg Utterance Duration = average time BETWEEN consecutive utterances
      (includes intervals both within turns AND between turns when speakers switch)

      Why can utterance duration be longer than

In [ ]:
# Examine the distribution of utterances_per_turn for Round 4 to understand the high std
round4_data = metrics_df[metrics_df['round'] == 4]['utterances_per_turn']
print("Round 4: Utterances per Turn Distribution")
print("="*60)
print(f"Mean: {round4_data.mean():.2f}")
print(f"Std:  {round4_data.std():.2f}")
print(f"Min:  {round4_data.min():.2f}")
print(f"Max:  {round4_data.max():.2f}")
print(f"Median: {round4_data.median():.2f}")
print(f"\nWhy can std be higher than mean?")
print("This indicates high variability and a right-skewed distribution.")
print("Some dialogs have very few turns with many utterances (high ratio),")
print("while others have many turns with few utterances (low ratio).")
print("\nSample values:")
print(round4_data.sort_values().head(10).tolist())
print("...")
print(round4_data.sort_values().tail(10).tolist())
